# GOase Plate Reader Processing

This notebook converts raw Tecan plate reader Excel exports into a structured GOase activity dataset. It reads all `.xlsx` files in `data/`, parses kinetic absorbance measurements, reshapes each plate from wide plate-reader format into tidy long format, merges available well metadata, and writes a reproducible CSV output.

The target output structure is inferred from `example_output.csv`.

## Input Files

Expected repository inputs:

- `data/Plate1.xlsx`
- `data/Plate2.xlsx`
- `data/Plate3.xlsx`
- `example_output.csv`

The Excel files are Tecan i-control exports with a `Raw` worksheet. Each raw sheet contains metadata rows followed by a kinetic measurement table where cycle numbers, elapsed times, and temperatures are stored in header rows and wells `A1` through `H12` are stored as rows.

The notebook discovers Excel files with `Path('data').glob('*.xlsx')`, so additional plates with the same Tecan layout can be added without changing the file list.

In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
from pandas.testing import assert_frame_equal

DATA_DIR = Path('data')
EXAMPLE_OUTPUT = Path('example_output.csv')
OUTPUT_CSV = Path('processed_GOase_activity_dataset.csv')

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)

## Inspect the Expected Output Format

`example_output.csv` defines the final column order and data types. The example file contains a known slice for `Plate1`, well `A01`; the full processed dataset will contain all detected plates, wells, and cycles.

In [2]:
example_df = pd.read_csv(EXAMPLE_OUTPUT)

print('Example output shape:', example_df.shape)
print('Example output columns:', list(example_df.columns))
print('\nExample dtypes:')
print(example_df.dtypes)

display(example_df.head())

Example output shape: (34, 7)
Example output columns: ['plate', 'well', 'cycle_nr', 'time_s', 'temperature_C', 'raw_value', 'mutant']

Example dtypes:
plate             object
well              object
cycle_nr           int64
time_s             int64
temperature_C    float64
raw_value        float64
mutant            object
dtype: object


,plate,well,cycle_nr,time_s,temperature_C,raw_value,mutant
0,Plate1,A01,1,0,24.8,0.2032,F313D
1,Plate1,A01,2,1200,24.5,0.1888,F313D
2,Plate1,A01,3,2400,25.5,0.1898,F313D
3,Plate1,A01,4,3600,25.5,0.1895,F313D
4,Plate1,A01,5,4800,25.7,0.1904,F313D


The required tidy output schema is:

- `plate`: plate identifier, derived from the Excel filename stem, for example `Plate1`.
- `well`: zero-padded 96-well coordinate, for example `A01`.
- `cycle_nr`: kinetic cycle number.
- `time_s`: elapsed measurement time in seconds.
- `temperature_C`: measured temperature in degrees Celsius.
- `raw_value`: raw absorbance measurement.
- `mutant`: well metadata. The raw Tecan files do not contain mutant assignments, so this notebook derives known assignments from `example_output.csv` and leaves missing assignments as `NA`.

## Locate Raw Excel Files

In [3]:
excel_files = sorted(DATA_DIR.glob('*.xlsx'))

print('Detected Excel files:')
for path in excel_files:
    print(f' - {path}')

if not excel_files:
    raise FileNotFoundError(f'No .xlsx files found in {DATA_DIR.resolve()}')

Detected Excel files:
 - data/Plate1.xlsx
 - data/Plate2.xlsx
 - data/Plate3.xlsx


## Parsing Helpers

The parsing logic below is specific to the Tecan raw export layout observed in these files:

- The kinetic table starts at the row labeled `Cycle Nr.`.
- The next required rows are `Time [s]` and `Temp. [°C]`.
- Well rows are labeled `A1` through `H12` in the first column.
- Measurement cycles are stored across columns, so the table is reshaped from wide to long format.

The functions validate the expected markers and skip sheets that do not contain the required raw kinetic table.

In [4]:
OUTPUT_COLUMNS = list(example_df.columns)
WELL_PATTERN = re.compile(r'^[A-Ha-h](?:[1-9]|1[0-2])$')


def normalize_well(value):
    """Convert Tecan well labels such as A1 to the output format A01."""
    match = re.fullmatch(r'([A-Ha-h])(\d{1,2})', str(value).strip())
    if not match:
        return pd.NA
    row, col = match.groups()
    return f'{row.upper()}{int(col):02d}'


def find_row_index(raw_df, label, startswith=False):
    """Find the first row whose first cell matches a required Tecan marker."""
    first_col = raw_df.iloc[:, 0].astype(str).str.strip()
    if startswith:
        matches = first_col[first_col.str.startswith(label, na=False)]
    else:
        matches = first_col[first_col.eq(label)]
    if matches.empty:
        raise ValueError(f'Missing required row label: {label}')
    return int(matches.index[0])


def select_raw_sheet(workbook_path):
    """Prefer the Raw sheet; otherwise fall back to the first sheet containing Cycle Nr."""
    excel = pd.ExcelFile(workbook_path)
    candidate_sheets = ['Raw'] if 'Raw' in excel.sheet_names else excel.sheet_names

    for sheet_name in candidate_sheets:
        raw = pd.read_excel(workbook_path, sheet_name=sheet_name, header=None)
        first_col = raw.iloc[:, 0].astype(str).str.strip()
        if first_col.eq('Cycle Nr.').any():
            return sheet_name, raw

    raise ValueError(f'No sheet with a Tecan kinetic table found in {workbook_path}')


def parse_tecan_plate(workbook_path):
    """Parse one Tecan plate reader workbook into tidy long-format measurements."""
    sheet_name, raw = select_raw_sheet(workbook_path)

    cycle_row = find_row_index(raw, 'Cycle Nr.')
    time_row = find_row_index(raw, 'Time [s]')
    temp_row = find_row_index(raw, 'Temp.', startswith=True)

    cycles = pd.to_numeric(raw.loc[cycle_row, 1:], errors='coerce')
    cycle_cols = cycles[cycles.notna()].index.tolist()
    if not cycle_cols:
        raise ValueError(f'No cycle columns detected in {workbook_path}')

    times = pd.to_numeric(raw.loc[time_row, cycle_cols], errors='coerce')
    temperatures = pd.to_numeric(raw.loc[temp_row, cycle_cols], errors='coerce')

    records = []
    for row_idx in range(temp_row + 1, raw.shape[0]):
        well_label = raw.iat[row_idx, 0]
        if not isinstance(well_label, str) or not WELL_PATTERN.fullmatch(well_label.strip()):
            continue

        well = normalize_well(well_label)
        values = pd.to_numeric(raw.loc[row_idx, cycle_cols], errors='coerce')

        for col in cycle_cols:
            if pd.isna(cycles[col]) or pd.isna(values[col]):
                continue

            # Tecan elapsed times contain decimal tenths from export precision. The example output stores integer seconds by
            # truncating the decimal portion rather than rounding it.
            time_value = pd.NA if pd.isna(times[col]) else int(float(times[col]))
            temp_value = pd.NA if pd.isna(temperatures[col]) else round(float(temperatures[col]), 1)

            records.append({
                'plate': workbook_path.stem,
                'well': well,
                'cycle_nr': int(cycles[col]),
                'time_s': time_value,
                'temperature_C': temp_value,
                'raw_value': float(values[col]),
            })

    parsed = pd.DataFrame.from_records(records)
    if parsed.empty:
        raise ValueError(f'No well measurements parsed from {workbook_path} sheet {sheet_name!r}')

    parsed = parsed.sort_values(['plate', 'well', 'cycle_nr']).reset_index(drop=True)
    print(f'{workbook_path.name}: sheet={sheet_name!r}, rows={len(parsed):,}, wells={parsed["well"].nunique()}, cycles={parsed["cycle_nr"].nunique()}')
    return parsed

## Mutant Metadata

The raw Excel files contain measurement values and run metadata, but no visible well-to-mutant map. To avoid introducing unsupported labels, the notebook extracts known `plate`/`well`/`mutant` assignments from `example_output.csv` and merges those onto the processed measurements.

For the current files, this provides the known assignment `Plate1` / `A01` / `F313D`. All other mutant values remain missing until a complete plate layout table is added or the example file is expanded.

In [5]:
def mutant_metadata_from_example(example):
    required = {'plate', 'well', 'mutant'}
    missing = required.difference(example.columns)
    if missing:
        raise ValueError(f'Example output is missing mutant metadata columns: {sorted(missing)}')

    metadata = (
        example[['plate', 'well', 'mutant']]
        .dropna(subset=['plate', 'well', 'mutant'])
        .drop_duplicates()
        .reset_index(drop=True)
    )

    duplicate_keys = metadata.duplicated(['plate', 'well'], keep=False)
    if duplicate_keys.any():
        duplicated = metadata.loc[duplicate_keys].sort_values(['plate', 'well'])
        raise ValueError('Conflicting mutant assignments found in example_output.csv:\n' + duplicated.to_string(index=False))

    return metadata


mutant_metadata = mutant_metadata_from_example(example_df)
print('Known mutant assignments derived from example_output.csv:')
display(mutant_metadata)

Known mutant assignments derived from example_output.csv:


,plate,well,mutant
0,Plate1,A01,F313D


## Process All Plates

In [6]:
plate_frames = []
for workbook_path in excel_files:
    try:
        plate_frames.append(parse_tecan_plate(workbook_path))
    except Exception as exc:
        print(f'WARNING: skipped {workbook_path}: {exc}')

if not plate_frames:
    raise RuntimeError('No plate data could be parsed from the detected Excel files.')

processed_df = pd.concat(plate_frames, ignore_index=True)
processed_df = processed_df.merge(mutant_metadata, on=['plate', 'well'], how='left')
processed_df = processed_df[OUTPUT_COLUMNS]
processed_df = processed_df.sort_values(['plate', 'well', 'cycle_nr']).reset_index(drop=True)

# Match the example schema closely: integer cycle/time columns, one decimal temperature precision, and raw float values.
processed_df['cycle_nr'] = processed_df['cycle_nr'].astype('int64')
processed_df['time_s'] = processed_df['time_s'].astype('int64')
processed_df['temperature_C'] = processed_df['temperature_C'].astype('float64')
processed_df['raw_value'] = processed_df['raw_value'].astype('float64')

print('Processed output shape:', processed_df.shape)
print('Processed output columns:', list(processed_df.columns))
print('\nRows per plate:')
print(processed_df.groupby('plate').size())
print('\nMissing mutant assignments:', int(processed_df['mutant'].isna().sum()))

display(processed_df.head())

Plate1.xlsx: sheet='Raw', rows=4,704, wells=96, cycles=49


Plate2.xlsx: sheet='Raw', rows=4,704, wells=96, cycles=49


Plate3.xlsx: sheet='Raw', rows=4,704, wells=96, cycles=49
Processed output shape: (14112, 7)
Processed output columns: ['plate', 'well', 'cycle_nr', 'time_s', 'temperature_C', 'raw_value', 'mutant']

Rows per plate:
plate
Plate1    4704
Plate2    4704
Plate3    4704
dtype: int64

Missing mutant assignments: 14063


,plate,well,cycle_nr,time_s,temperature_C,raw_value,mutant
0,Plate1,A01,1,0,24.8,0.2032,F313D
1,Plate1,A01,2,1200,24.5,0.1888,F313D
2,Plate1,A01,3,2400,25.5,0.1898,F313D
3,Plate1,A01,4,3600,25.5,0.1895,F313D
4,Plate1,A01,5,4800,25.7,0.1904,F313D


## Validation Against the Example Output

Validation checks confirm that:

1. The generated column order matches `example_output.csv`.
2. The generated data types are compatible with the example schema.
3. The known example slice is reproduced from the raw Excel data within floating-point tolerance.
4. Missing mutant assignments are reported explicitly.

In [7]:
print('Column match:', list(processed_df.columns) == list(example_df.columns))
assert list(processed_df.columns) == list(example_df.columns), 'Output columns do not match example_output.csv'

print('\nGenerated dtypes:')
print(processed_df.dtypes)

print('\nExample dtypes:')
print(example_df.dtypes)

key_columns = ['plate', 'well', 'cycle_nr']
example_keys = example_df[key_columns].drop_duplicates()
example_slice = example_df.sort_values(key_columns).reset_index(drop=True)
processed_slice = (
    processed_df.merge(example_keys, on=key_columns, how='inner')
    .sort_values(key_columns)
    .reset_index(drop=True)
)

# Compare using tolerance because Excel and CSV float serialization can differ by tiny binary floating-point amounts.
assert_frame_equal(
    processed_slice[example_df.columns],
    example_slice[example_df.columns],
    check_dtype=False,
    check_exact=False,
    rtol=1e-7,
    atol=1e-9,
)

print(f'Example slice validation passed for {len(example_slice)} rows.')
print(f'Full processed dataset contains {len(processed_df):,} rows.')
print(f'Mutant metadata missing for {processed_df["mutant"].isna().sum():,} rows; add a complete plate layout if those labels are required.')

Column match: True

Generated dtypes:
plate             object
well              object
cycle_nr           int64
time_s             int64
temperature_C    float64
raw_value        float64
mutant            object
dtype: object

Example dtypes:
plate             object
well              object
cycle_nr           int64
time_s             int64
temperature_C    float64
raw_value        float64
mutant            object
dtype: object
Example slice validation passed for 34 rows.
Full processed dataset contains 14,112 rows.
Mutant metadata missing for 14,063 rows; add a complete plate layout if those labels are required.


## Write Processed CSV

In [8]:
processed_df.to_csv(OUTPUT_CSV, index=False)
print(f'Wrote {OUTPUT_CSV} with shape {processed_df.shape}')

display(pd.read_csv(OUTPUT_CSV).head())

Wrote processed_GOase_activity_dataset.csv with shape (14112, 7)


,plate,well,cycle_nr,time_s,temperature_C,raw_value,mutant
0,Plate1,A01,1,0,24.8,0.2032,F313D
1,Plate1,A01,2,1200,24.5,0.1888,F313D
2,Plate1,A01,3,2400,25.5,0.1898,F313D
3,Plate1,A01,4,3600,25.5,0.1895,F313D
4,Plate1,A01,5,4800,25.7,0.1904,F313D


## Assumptions and Notes

- The workbook stem, such as `Plate1`, is used as the `plate` identifier.
- The Tecan sheet is expected to contain `Cycle Nr.`, `Time [s]`, and `Temp. [°C]` rows.
- Well names are normalized from `A1` to `A01` to match the example output.
- Tecan elapsed times are exported with decimal tenths; the example output stores integer seconds by truncating those decimals.
- The raw Excel workbooks do not include mutant assignments. Known assignments are derived from `example_output.csv`; currently only `Plate1` / `A01` / `F313D` is available.
- Raw Excel files are read only. The notebook writes only `processed_GOase_activity_dataset.csv`.